# CLI v3 — Noisy-Label Filter Using NASA-TLX

**Project:** Predicting and Preventing Burnout Using Cognitive Load Analysis
**Notebook:** 04 — conservative label quality improvement

---

## What this notebook does

Notebooks 01 and 02 classify cognitive load using **design-based labels**:
- `*_easy` tasks → Low Load
- `*_hard` tasks → High Load

This labeling is objective (based on task design) but not validated by participant
experience. NASA-TLX questionnaire data (from notebook 03) showed that some
participants found "easy" tasks genuinely demanding and some found "hard" tasks
surprisingly light.

**v3 strategy:** Keep the original design labels, but remove segments where the
participant's NASA-TLX score strongly contradicts the label. This is a *conservative
filter* — we do not replace labels, we only remove the most ambiguous training examples.

### Noisy-label filter rules

| Design label | Weighted NASA Score | Action |
|---|---|---|
| Low (easy tasks) | > 60 | Remove — participant found the task too demanding for Low label |
| High (hard tasks) | < 40 | Remove — participant found the task too easy for High label |

Segments with no NASA-TLX match are **kept** — when in doubt, do not filter.

### What stays the same as v2
- Lab1 + Lab2 sessions only
- EDA + HRV + TEMP features only (no EEG, no Raw)
- LOPO cross-validation with per-participant z-score normalization
- Gradient Boosting classifier (best model from v2)
- Segment-level evaluation

### What changes
- Some (participant, session, segment) triplets are removed before training
- Both LOPO and segment-level results are recomputed on the filtered dataset
- A direct v2 vs v3 comparison table is produced

In [ ]:
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    roc_auc_score, recall_score, roc_curve, auc as sklearn_auc,
)

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_ROOT     = Path('.')
SESSIONS      = ['Lab1', 'Lab2']
FEATURE_FILES = ['EDA_features', 'HRV_features', 'TEMP_features']

LOW_EXACT  = {'relaxation_video', 'video_baseline'}
LOW_SUFFIX  = '_easy'
HIGH_SUFFIX = '_hard'

# Noisy-label filter thresholds
NASA_LOW_MAX  = 40   # High-label segments with NASA < 40  are removed
NASA_HIGH_MIN = 60   # Low-label segments  with NASA > 60  are removed

# Best model from v2 (Gradient Boosting)
BASE_MODEL = GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_SEED)

print('Imports OK.')
print(f'NASA filter: remove Low segments with NASA > {NASA_HIGH_MIN}')
print(f'             remove High segments with NASA < {NASA_LOW_MAX}')

## Step 1 — Load Feature Dataset (same as v2)

The feature loader is identical to notebooks 01 and 02. Labels are assigned
by design rules only — NASA-TLX is not used here.
The full unfiltered dataset is loaded first; filtering happens after NASA matching.

In [ ]:
def segment_to_label(segment_name):
    name = segment_name.lower()
    if name in LOW_EXACT or name.endswith(LOW_SUFFIX):
        return 0, 'Low'
    if name.endswith(HIGH_SUFFIX):
        return 1, 'High'
    return None, None


def load_segment(seg_dir, pid, session):
    seg_name = seg_dir.name
    label, load_level = segment_to_label(seg_name)
    if label is None:
        return None, f"unrecognized segment '{seg_name}'"
    dfs = []
    for feat_name in FEATURE_FILES:
        fpath = seg_dir / f'{feat_name}.pickle'
        if not fpath.exists():
            return None, f'{feat_name}.pickle missing'
        with open(fpath, 'rb') as f:
            feat_df = pickle.load(f)
        if not isinstance(feat_df, pd.DataFrame):
            feat_df = pd.DataFrame(feat_df)
        dfs.append(feat_df.reset_index(drop=True))
    combined = pd.concat(dfs, axis=1)
    combined['label']          = label
    combined['load_level']     = load_level
    combined['participant_id'] = pid
    combined['session']        = session
    combined['segment']        = seg_name
    combined['window_idx']     = range(len(combined))
    return combined, None


def load_all_features():
    all_segs = []
    for pdir in sorted(DATA_ROOT.glob('UN_*')):
        if not pdir.is_dir():
            continue
        for session in SESSIONS:
            fp = pdir / session / 'Features'
            if not fp.exists():
                continue
            for sd in sorted(d for d in fp.iterdir() if d.is_dir()):
                df, _ = load_segment(sd, pdir.name, session)
                if df is not None:
                    all_segs.append(df)
    return pd.concat(all_segs, ignore_index=True)


dataset = load_all_features()

META_COLS    = ['label', 'load_level', 'participant_id', 'session', 'segment', 'window_idx']
FEATURE_COLS = [c for c in dataset.columns if c not in META_COLS]

# Fill NaN windows with global median (same as v2)
nan_count = dataset[FEATURE_COLS].isnull().sum().sum()
dataset[FEATURE_COLS] = dataset[FEATURE_COLS].fillna(dataset[FEATURE_COLS].median())

print(f'Dataset (full, unfiltered): {dataset.shape[0]:,} windows, {len(FEATURE_COLS)} features')
print(f'Participants: {dataset["participant_id"].nunique()}')
print(f'NaN cells filled: {nan_count}')
print()
print('Class balance (unfiltered):')
print(dataset['load_level'].value_counts().to_frame('windows'))

## Step 2 — Load Lab Task_Labels.csv (NASA-TLX Scores)

We load the NASA-TLX questionnaire data for Lab1 and Lab2 sessions.
Only two columns are needed: `Task` (segment name) and `Weighted Nasa Score`.

### Segment name mapping

The Features folder uses the name `relaxation_video` for the baseline rest segment.
The Task_Labels.csv uses `video_baseline` for the same segment.
We apply one explicit mapping before matching:

```
relaxation_video  →  video_baseline
```

All other segment names are identical in both the Features folders and Task_Labels.csv.

In [ ]:
LAB_KEEP = ['Task', 'Weighted Nasa Score']

# relaxation_video (Features folder name) -> video_baseline (Task_Labels name)
SEGMENT_NAME_MAP = {
    'relaxation_video': 'video_baseline',
}

def normalize_segment_name(name):
    """Map Features folder segment name to Task_Labels.csv Task name."""
    return SEGMENT_NAME_MAP.get(name.lower(), name.lower())


nasa_frames = []
nasa_load_log = []

for pdir in sorted(DATA_ROOT.glob('UN_*')):
    pid = pdir.name
    for session in SESSIONS:
        fpath = pdir / session / 'Task_Labels.csv'
        if not fpath.exists():
            nasa_load_log.append({'participant_id': pid, 'session': session,
                                   'status': 'missing'})
            continue
        try:
            df = pd.read_csv(fpath)
            present = [c for c in LAB_KEEP if c in df.columns]
            if 'Weighted Nasa Score' not in present:
                nasa_load_log.append({'participant_id': pid, 'session': session,
                                       'status': 'missing column'})
                continue
            df = df[present].copy()
            df['participant_id'] = pid
            df['session']        = session
            df['task_key']       = df['Task'].str.lower().str.strip()
            df['Weighted Nasa Score'] = pd.to_numeric(
                df['Weighted Nasa Score'], errors='coerce'
            )
            nasa_frames.append(df)
            nasa_load_log.append({'participant_id': pid, 'session': session,
                                   'status': 'ok', 'n_tasks': len(df)})
        except Exception as e:
            nasa_load_log.append({'participant_id': pid, 'session': session,
                                   'status': f'error: {e}'})

nasa_df = pd.concat(nasa_frames, ignore_index=True) if nasa_frames else pd.DataFrame()
log_df  = pd.DataFrame(nasa_load_log)

n_ok      = (log_df['status'] == 'ok').sum()
n_missing = (log_df['status'] != 'ok').sum()
print(f'Task_Labels.csv files loaded: {n_ok}  |  missing/error: {n_missing}')
print(f'Total (participant, session, task) rows: {len(nasa_df)}')
print()
print('NASA score range:', nasa_df['Weighted Nasa Score'].min(),
      '–', nasa_df['Weighted Nasa Score'].max())
print('Missing NASA values:', nasa_df['Weighted Nasa Score'].isna().sum())
print()
if n_missing > 0:
    print('Files not loaded:')
    print(log_df[log_df['status'] != 'ok'][['participant_id','session','status']]
          .to_string(index=False))

## Step 3 — Match Segments to NASA Scores

For each segment in the feature dataset we look up its NASA-TLX score using the key:

    (participant_id, session, normalized_segment_name)

The normalization applies the `relaxation_video → video_baseline` mapping before joining.

### Conservative filtering policy

- Segments **with a NASA match** and a contradicting score → removed
- Segments **with a NASA match** and an agreeing score → kept
- Segments **with no NASA match** (e.g. missing Task_Labels.csv file) → **kept**

"When in doubt, keep" — we do not penalize participants whose questionnaire data
is unavailable. This makes the filter conservative and avoids bias from
differential data availability.

In [ ]:
# Build lookup: (participant_id, session, task_key) -> Weighted NASA Score
nasa_lookup = (
    nasa_df.dropna(subset=['Weighted Nasa Score'])
           .set_index(['participant_id', 'session', 'task_key'])['Weighted Nasa Score']
           .to_dict()
)
print(f'NASA lookup entries: {len(nasa_lookup)}')

# Get unique segments from the dataset
seg_index = (
    dataset.drop_duplicates(subset=['participant_id', 'session', 'segment'])
           [['participant_id', 'session', 'segment', 'label', 'load_level']]
           .copy()
)
seg_index['task_key'] = seg_index['segment'].apply(normalize_segment_name)
seg_index['nasa_score'] = seg_index.apply(
    lambda r: nasa_lookup.get((r['participant_id'], r['session'], r['task_key'])),
    axis=1
)

n_segs_total    = len(seg_index)
n_matched       = seg_index['nasa_score'].notna().sum()
n_unmatched     = seg_index['nasa_score'].isna().sum()

print(f'Total segments in dataset : {n_segs_total}')
print(f'Segments with NASA match  : {n_matched}  ({100*n_matched/n_segs_total:.1f}%)')
print(f'Segments without NASA match (kept): {n_unmatched}  ({100*n_unmatched/n_segs_total:.1f}%)')
print()

# Apply filter rules
is_noisy_low  = (
    (seg_index['label'] == 0) &
    seg_index['nasa_score'].notna() &
    (seg_index['nasa_score'] > NASA_HIGH_MIN)
)
is_noisy_high = (
    (seg_index['label'] == 1) &
    seg_index['nasa_score'].notna() &
    (seg_index['nasa_score'] < NASA_LOW_MAX)
)
is_noisy = is_noisy_low | is_noisy_high

n_noisy_low  = is_noisy_low.sum()
n_noisy_high = is_noisy_high.sum()
n_noisy_total = is_noisy.sum()

print(f'Noisy Low segments  (Low label, NASA > {NASA_HIGH_MIN}): {n_noisy_low}')
print(f'Noisy High segments (High label, NASA < {NASA_LOW_MAX}): {n_noisy_high}')
print(f'Total noisy segments to remove: {n_noisy_total}')
print()

# Build set of (participant, session, segment) to remove
noisy_keys = set(
    zip(
        seg_index.loc[is_noisy, 'participant_id'],
        seg_index.loc[is_noisy, 'session'],
        seg_index.loc[is_noisy, 'segment'],
    )
)

# Filter dataset at window level
keep_mask = ~dataset.apply(
    lambda r: (r['participant_id'], r['session'], r['segment']) in noisy_keys,
    axis=1
)
dataset_v3 = dataset[keep_mask].copy()

n_windows_v2 = len(dataset)
n_windows_v3 = len(dataset_v3)
n_removed    = n_windows_v2 - n_windows_v3

print('=' * 55)
print('FILTERING RESULT')
print('=' * 55)
print(f'Windows before filter (v2): {n_windows_v2:,}')
print(f'Windows after filter  (v3): {n_windows_v3:,}  ({100*n_windows_v3/n_windows_v2:.1f}% retained)')
print(f'Windows removed           : {n_removed:,}  ({100*n_removed/n_windows_v2:.1f}%)')
print(f'Segments removed          : {n_noisy_total}')
print()
print('Class balance BEFORE filter (v2):')
print(dataset['load_level'].value_counts().to_frame('windows'))
print()
print('Class balance AFTER filter (v3):')
print(dataset_v3['load_level'].value_counts().to_frame('windows'))

In [ ]:
# Show which segments were removed
noisy_seg_df = seg_index[is_noisy][
    ['participant_id', 'session', 'segment', 'load_level', 'nasa_score']
].sort_values(['load_level', 'nasa_score'], ascending=[True, False]).reset_index(drop=True)

print('Removed segments (noisy labels):')
display(noisy_seg_df)
print()

# Summary by task type
print('Removed segments by task name:')
print(noisy_seg_df['segment'].value_counts().to_frame('count_removed'))

## Why We Filter Rather Than Replace Labels

### Option A: replace design labels with NASA-TLX labels
Replace `label = 0` with `label = 1` if NASA > 60, and `label = 1` with `label = 0`
if NASA < 40. This would make labels more subjectively valid.

**Why we do not do this yet:**
1. **Risk of circular noise:** replacing one form of label noise with another.
   A high NASA-TLX score on an easy task might reflect anxiety, not cognitive load.
2. **Small N consequence:** changing even a few labels per participant can
   substantially shift the class distribution within a LOPO fold.
3. **Unverified assumption:** we do not yet know whether NASA-TLX predicts
   physiological signals better than design labels.

### Option B: remove contradicting segments (this notebook)
Remove windows from segments where design label strongly disagrees with NASA-TLX.
The model then trains only on examples where the label is relatively unambiguous.

**Why this is safer:**
1. We are not asserting a new label — we are only removing uncertain examples.
2. If the filter improves performance, it confirms that label noise was degrading v2.
3. If performance does not improve, it suggests that noise rate was too low to matter,
   or that the model already handles ambiguous examples robustly.

### Conservative filter threshold choice
We use NASA > 60 (not 50) and NASA < 40 (not 50) to target only the clearest
disagreements. Using 50 as a threshold would remove too many borderline segments
and risk discarding genuinely informative examples near the boundary.

## Step 4 — Per-Participant Z-Score Normalization

Same normalization as v2: each participant's features are z-scored using their own
mean and standard deviation. Applied separately to both the v2 (full) and v3
(filtered) datasets so the comparison is fair.

In [ ]:
def normalize_per_participant(df, feature_cols):
    out = df.copy()
    for pid, grp in out.groupby('participant_id'):
        mu  = grp[feature_cols].mean()
        sig = grp[feature_cols].std().replace(0, 1)
        out.loc[grp.index, feature_cols] = (grp[feature_cols] - mu) / sig
    return out


dataset_v2_norm = normalize_per_participant(dataset,    FEATURE_COLS)
dataset_v3_norm = normalize_per_participant(dataset_v3, FEATURE_COLS)

print('Normalization complete.')
print(f'v2 normalized: {dataset_v2_norm.shape[0]:,} windows')
print(f'v3 normalized: {dataset_v3_norm.shape[0]:,} windows')

## Step 5 — LOPO Cross-Validation

We run the same 24-fold LOPO protocol as v2 on both datasets.
Each fold: train on 23 participants, test on 1, fresh model every fold.

**Key LOPO detail for v3:** When participant P is the test fold,
- Training set = windows from the other 23 participants (after filtering in v3)
- Test set = ALL of participant P's windows (no filtering of test data)

We never filter the test set — the filter only affects training data.
This ensures we are evaluating the model on the same test windows in both v2 and v3,
making the comparison valid.

In [ ]:
def run_lopo_combined(df_norm, feature_cols, label='', test_df_norm=None):
    """
    Single LOPO pass: returns fold metrics, stacked predictions,
    AND a window-level DataFrame for segment aggregation.

    test_df_norm : optional DataFrame used as the SOURCE for test data.
        When None  — train and test both come from df_norm (v2 behaviour).
        When given — train comes from df_norm, test comes from test_df_norm.
        Used for v3: train on filtered data, test on the same full windows as v2.
    """
    participants = sorted(df_norm['participant_id'].unique())
    fold_records, window_records = [], []

    test_src = test_df_norm if test_df_norm is not None else df_norm

    if label:
        print(f'LOPO ({label}): {len(participants)} folds ...')

    for k, test_pid in enumerate(participants):
        train_mask = df_norm['participant_id'] != test_pid
        test_mask  = test_src['participant_id'] == test_pid

        X_tr = df_norm.loc[train_mask, feature_cols].values
        y_tr = df_norm.loc[train_mask, 'label'].values
        X_te = test_src.loc[test_mask,  feature_cols].values
        y_te = test_src.loc[test_mask,  'label'].values

        model = clone(BASE_MODEL)
        model.fit(X_tr, y_tr)
        y_proba = model.predict_proba(X_te)[:, 1]
        y_pred  = (y_proba >= 0.5).astype(int)

        try:
            auc_val = roc_auc_score(y_te, y_proba)
        except ValueError:
            auc_val = np.nan

        fold_records.append({
            'participant':  test_pid,
            'accuracy':     round(accuracy_score(y_te, y_pred), 3),
            'f1_macro':     round(f1_score(y_te, y_pred, average='macro',
                                           zero_division=0), 3),
            'roc_auc':      round(auc_val, 3) if not np.isnan(auc_val) else np.nan,
            'balanced_acc': round(balanced_accuracy_score(y_te, y_pred), 3),
            'recall_high':  round(recall_score(y_te, y_pred, pos_label=1,
                                               zero_division=0), 3),
        })

        meta = test_src.loc[test_mask,
                            ['participant_id','session','segment','label']].copy()
        meta['proba_high'] = y_proba
        window_records.append(meta)

        if label and (k + 1) % 6 == 0:
            print(f'  fold {k+1}/{len(participants)} done')

    fold_df    = pd.DataFrame(fold_records)
    window_df  = pd.concat(window_records, ignore_index=True)
    y_true_all  = window_df['label'].values
    y_proba_all = window_df['proba_high'].values

    return fold_df, y_true_all, y_proba_all, window_df


def segment_level_metrics(window_df, threshold=0.5):
    seg = (
        window_df
        .groupby(['participant_id','session','segment'], sort=False)
        .agg(proba_mean=('proba_high','mean'), true_label=('label','first'))
        .reset_index()
    )
    y_true  = seg['true_label'].values
    y_proba = seg['proba_mean'].values
    y_pred  = (y_proba >= threshold).astype(int)
    try:
        auc_val = roc_auc_score(y_true, y_proba)
    except ValueError:
        auc_val = np.nan
    return {
        'n_segments':   len(seg),
        'accuracy':     round(accuracy_score(y_true, y_pred), 3),
        'balanced_acc': round(balanced_accuracy_score(y_true, y_pred), 3),
        'f1_macro':     round(f1_score(y_true, y_pred, average='macro',
                                       zero_division=0), 3),
        'roc_auc':      round(auc_val, 3) if not np.isnan(auc_val) else np.nan,
        'recall_high':  round(recall_score(y_true, y_pred, pos_label=1,
                                           zero_division=0), 3),
    }, seg


print('LOPO functions defined.')

## Step 6 — Run LOPO: v2 Original vs v3 Filtered

Both runs use the same GBT model and same LOPO protocol.
This may take 2–4 minutes.

In [ ]:
print('Running LOPO on v2 (full dataset) — 24 folds ...')
fold_v2, y_true_v2, y_proba_v2, win_v2 = run_lopo_combined(
    dataset_v2_norm, FEATURE_COLS, label='v2'
)
print()

# v3: train on filtered data, test on the SAME full windows as v2
# (test_df_norm=dataset_v2_norm ensures the test participant is unfiltered)
print('Running LOPO on v3 (noisy-filtered training, unfiltered test) — 24 folds ...')
fold_v3, y_true_v3, y_proba_v3, win_v3 = run_lopo_combined(
    dataset_v3_norm, FEATURE_COLS, label='v3',
    test_df_norm=dataset_v2_norm
)
print()

def fold_summary(fold_df, label):
    print(f'--- {label} ---')
    for col in ['accuracy','balanced_acc','f1_macro','roc_auc','recall_high']:
        vals = fold_df[col].dropna()
        print(f'  {col:<15}: {vals.mean():.3f} +/- {vals.std():.3f}')
    print()

fold_summary(fold_v2, 'v2 Original (full dataset)')
fold_summary(fold_v3, 'v3 Noisy-Filtered (unfiltered test)')

seg_metrics_v2, seg_df_v2 = segment_level_metrics(win_v2)
seg_metrics_v3, seg_df_v3 = segment_level_metrics(win_v3)
print(f'Segment-level evaluation (both on same unfiltered test windows):')
print(f'  v2: {seg_metrics_v2["n_segments"]} segments | Segment AUC: {seg_metrics_v2["roc_auc"]:.3f}')
print(f'  v3: {seg_metrics_v3["n_segments"]} segments | Segment AUC: {seg_metrics_v3["roc_auc"]:.3f}')
print()
print('Note: v2 and v3 segment counts should be equal (same test windows).')

## Step 7 — v2 vs v3 Comparison

In [ ]:
def make_row(label, fold_df, seg_metrics, y_true, y_proba, n_windows, n_train_segs):
    fd = fold_df.dropna(subset=['roc_auc'])
    return {
        'Variant':         label,
        'Train Windows':   f'{n_windows:,}',
        'Train Segs':      n_train_segs,
        'Accuracy':        round(fd['accuracy'].mean(), 3),
        'Balanced Acc':    round(fd['balanced_acc'].mean(), 3),
        'F1 (macro)':      round(fd['f1_macro'].mean(), 3),
        'LOPO AUC':        round(fd['roc_auc'].mean(), 3),
        'Recall High':     round(fd['recall_high'].mean(), 3),
        'Segment AUC':     seg_metrics['roc_auc'],
        'Seg Recall High': seg_metrics['recall_high'],
    }

n_segs_v2 = dataset.drop_duplicates(
    subset=['participant_id','session','segment']).shape[0]
n_segs_v3 = dataset_v3.drop_duplicates(
    subset=['participant_id','session','segment']).shape[0]

rows = [
    make_row('v2 — Original (full)',   fold_v2, seg_metrics_v2,
             y_true_v2, y_proba_v2, len(dataset),    n_segs_v2),
    make_row('v3 — Noisy-filter (NASA>60 / <40)', fold_v3, seg_metrics_v3,
             y_true_v3, y_proba_v3, len(dataset_v3), n_segs_v3),
]

comp_df = pd.DataFrame(rows).set_index('Variant')
print('=' * 80)
print('v2 vs v3 COMPARISON TABLE')
print('(Train Windows / Train Segs = training data; evaluation on same test windows)')
print('=' * 80)
display(comp_df)
print()

# Delta row
numeric_cols = ['Accuracy','Balanced Acc','F1 (macro)','LOPO AUC','Recall High',
                'Segment AUC','Seg Recall High']
for col in numeric_cols:
    v2_val = float(comp_df.loc['v2 — Original (full)', col])
    v3_val = float(comp_df.loc['v3 — Noisy-filter (NASA>60 / <40)', col])
    delta  = v3_val - v2_val
    sign   = '+' if delta >= 0 else ''
    print(f'  {col:<18}: v2={v2_val:.3f}  v3={v3_val:.3f}  delta={sign}{delta:.3f}')

In [ ]:
# Per-participant AUC: v2 vs v3
participants_sorted = sorted(fold_v2['participant'].values)
x = np.arange(len(participants_sorted))
width = 0.35

fd_v2 = fold_v2.set_index('participant')
fd_v3 = fold_v3.set_index('participant')

auc_v2 = [fd_v2.loc[p, 'roc_auc'] if p in fd_v2.index else np.nan
           for p in participants_sorted]
auc_v3 = [fd_v3.loc[p, 'roc_auc'] if p in fd_v3.index else np.nan
           for p in participants_sorted]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - width/2, auc_v2, width=width, color='steelblue', alpha=0.75,
       label='v2 Original', edgecolor='white')
ax.bar(x + width/2, auc_v3, width=width, color='tomato',    alpha=0.75,
       label='v3 Noisy-filter', edgecolor='white')
ax.axhline(0.5, color='black', linestyle='--', lw=1.2, label='Chance (0.5)')
ax.set_xticks(x)
ax.set_xticklabels([p.replace('UN_','P') for p in participants_sorted],
                   rotation=45, ha='right', fontsize=9)
ax.set_ylabel('LOPO ROC-AUC')
ax.set_title('Per-Participant LOPO ROC-AUC: v2 Original vs v3 Noisy-Filter')
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig('v2_vs_v3_participant_auc.png', dpi=150, bbox_inches='tight')
plt.show()

# Participants where v3 improved
improved = [(p, auc_v2[i], auc_v3[i], round(auc_v3[i]-auc_v2[i],3))
            for i, p in enumerate(participants_sorted)
            if not np.isnan(auc_v2[i]) and not np.isnan(auc_v3[i])
            and auc_v3[i] > auc_v2[i]]
declined = [(p, auc_v2[i], auc_v3[i], round(auc_v3[i]-auc_v2[i],3))
            for i, p in enumerate(participants_sorted)
            if not np.isnan(auc_v2[i]) and not np.isnan(auc_v3[i])
            and auc_v3[i] < auc_v2[i]]
print(f'Participants where v3 improved: {len(improved)}')
print(f'Participants where v3 declined: {len(declined)}')

In [ ]:
# Aggregated ROC curves: v2 vs v3 (window-level and segment-level)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: window-level ROC
for y_true, y_proba, label, color in [
    (y_true_v2, y_proba_v2, 'v2 Original',      'steelblue'),
    (y_true_v3, y_proba_v3, 'v3 Noisy-filter',  'tomato'),
]:
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    a = sklearn_auc(fpr, tpr)
    axes[0].plot(fpr, tpr, color=color, lw=2, label=f'{label}  (AUC={a:.3f})')

axes[0].plot([0, 1], [0, 1], '--', color='gray', lw=1, label='Chance')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('Window-Level ROC Curve (LOPO aggregated)')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# Right: segment-level ROC
for seg_df_plot, label, color in [
    (seg_df_v2, 'v2 Original',     'steelblue'),
    (seg_df_v3, 'v3 Noisy-filter', 'tomato'),
]:
    fpr_s, tpr_s, _ = roc_curve(
        seg_df_plot['true_label'], seg_df_plot['proba_mean']
    )
    a_s = sklearn_auc(fpr_s, tpr_s)
    axes[1].plot(fpr_s, tpr_s, color=color, lw=2,
                 label=f'{label}  (AUC={a_s:.3f})')

axes[1].plot([0, 1], [0, 1], '--', color='gray', lw=1, label='Chance')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Segment-Level ROC Curve (LOPO aggregated)')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.suptitle('v2 vs v3: ROC Curves', fontsize=11)
plt.tight_layout()
plt.savefig('v2_vs_v3_roc.png', dpi=150, bbox_inches='tight')
plt.show()

## Conclusion — Interpreting the v2 vs v3 Comparison

---

### If v3 performance is better than v2

An improvement in LOPO AUC or segment-level AUC means:

1. **Label noise was real and measurable.** Some segments labeled Low/High by design
   were physiologically indistinguishable because the participant's subjective
   experience contradicted the design label. Removing them gave the model
   cleaner training signal.

2. **NASA-TLX is a valid label quality signal.** The filter threshold
   (NASA > 60 for Low, NASA < 40 for High) identified genuinely ambiguous examples.
   This motivates using NASA-TLX more aggressively in a future v4 (e.g., as a soft
   weight rather than a binary filter).

3. **The gain is expected to be modest.** The number of noisy segments is small
   relative to the full dataset (~407 segments). A large gain would be surprising
   and should be verified; a small gain (+0.01 to +0.03 AUC) is the realistic range.

---

### If v3 performance is similar to or worse than v2

No improvement means:

1. **The noise rate was too low to matter.** With fewer than ~10% of segments
   filtered, the model's random forest / gradient boosting trees are robust
   enough to handle the noise without explicit filtering.

2. **The filter removed informative examples.** Segments where a participant
   found an easy task hard may carry *extra* physiological signal — the model
   may have used those "contradicting" patterns to generalize better.

3. **The label quality problem is structural, not local.** The fundamental
   issue is that design difficulty ≠ experienced difficulty for many participants.
   A binary filter on clear outliers does not address the underlying continuous
   mismatch. This motivates using NASA-TLX as a continuous weight (v4 direction).

---

### What to do next (v4 directions)

| Direction | What it does | Expected effort |
|---|---|---|
| NASA confidence weighting | Weight each sample by how strongly NASA agrees with the design label | Low — add `sample_weight` to GBT `.fit()` |
| NASA fixed-threshold labels | Replace design labels entirely using NASA cutoffs (< 40 = Low, > 60 = High) | Medium — redefines the learning problem |
| Wild + NASA labels | Include Wild sessions with NASA-derived labels | High — requires careful label assignment |
| Personalized calibration | Fine-tune per-person model using Lab data, test on Wild | High — changes the evaluation architecture |